# RAG进阶————切片、重排、混合搜索

基础的RAG检索top-k个最相关的片段，这对简单的问题管用。但是对于多跳问题、模糊的查询以及大体量的语料失效。RAG进阶就是为了解决在百万级别文档上而非小体量Demo上工作的。

## 问题描述

- 模糊的查询。
- 多跳问题。
- 超大规模语料。

基础RAG失败的原因在于向量相似度不完全等价于相关度。一个切片可能在毫不相关的问题上语义相关，比如你问赔偿金额多少，它讲的是赔偿政策。RAG进阶使用四种技法来处理：混合搜索（包含关键词搜索）、重排（更细致的打分）、变换查询（查询前明确目标）、更好的切片（以更合适的粒度）。

## 基本概念

### 混合搜索————语义+关键词

语义搜索能更好地理解含义。

关键词搜索则注重精确匹配。

同时跑两者，然后把结果融合起来。

比较出名的关键词检索算法，BM25（best matching 25）

### 倒数排名融合（Reciprocal Rank Fusion， RRF）

```
RRF_score(d) = sum over rankings R: 1 / (k + rank_R(d))
```

平衡两者信号。一篇在两种检索方法中都得高分的文档最后得分也高，而一篇在检索方法A中拿了第一高，方法B中检索不到的时候，得分一般。

### 重排

检索（无论是向量、关键词还是混合）很快但是不准确。使用双编码器：查询和每个文档分别做嵌入编码，然后再比较。

重排使用交叉编码器：查询和文档组合在一起送入大模型做相关性打分。模型能够同时看到二者，所以能捕捉更细粒度的关联。

权衡：交叉编码器比双编码器慢100-1000倍，经验型的做法是先用双编码器召回50个文档，然后用交叉编码器找出相关性最高的5个文档。

### 查询变换

有时候问题不出在检索，而在查询提问本身。比如模糊不清导致既捕捉不到关键词，嵌入向量也很模糊。

#### 重写查询

用LLM将用户输入的问题重新写一下，**用更好的方式**。

#### 假设性文档嵌入（Hypotheical Document Embeddings ———— HyDE）

先生成一个假设性的答案，嵌入它，然后再去检索。
```
Query: "..."
Hypotheical answer: "..."
```
直觉是：相对于原始问题，假设性的答案在嵌入空间中会离真实答案更近。

多一次LLM生成，换来更准确的检索结果。

### 父子切片

标准的切片逼你做权衡：小的切片检索精确，大的切片包含足够的上下文。解决方案是父子切片。

小的切片用于检索，但是检索到之后，将它的父切片返回到提示词中。这样，小的切片提供查询准确度，它的父切片又能提供足够的上下文。

### 元数据过滤

在跑向量搜索之前，将语料按元数据过滤。包括日期、来源、分类、作者、语言等。这些内容能够降低搜索范围以及防止不相关的结果。

生产级的RAG系统为每个切片存储元数据，向量数据库支持按照元数据在搜索之前过滤，在大尺度性能上很关键。

### 评估

三个指标评估RAG系统：
- 检索相关性。  实际关联文档出现在top-k中的概率是多少。  
- 忠诚度。  生成的答案多大程度上遵守了召回的结果。
- 答案正确性。  端到端的指标，结合了检索质量和生成质量。

忠诚度的检查方法：对答案中的每项声明，验证它是否出现在了召回的切片中。如果没有，那可能是幻觉。

# 开始编码

In [1]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    scores = {}
    for ranked_list in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked_list):
            if doc_id not in scores:
                scores[doc_id] = 0.0
            scores[doc_id] += 1.0 / (k + rank + 1)
    fused = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    return fused

In [ ]:
def create_parent_child_chunks(text, parent_size=200, child_size=50):
    words = text.split()
    parents = []
    children = []
    child_to_parent = {}

    parent_idx = 0
    start = 0

    while start < len(words):
        parent_end = min(start + parent_size, len(words))
        parent_text = " ".join(words[start:parent_end])
        parents.append(parent_text)

        child_start = start
        while child_start < parent_end:
            child_end = min(child_start + child_size, parent_end)
            child_text = " ".join(words[child_start:child_end])
            child_idx = len(children)
            children.append(child_text)
            child_to_parent[child_idx] = parent_idx
            child_start += child_size
        parent_idx += 1
        start += parent_size

    return parents, children, child_to_parent


## 生产级别重排

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, chunks, top_k=5):
    pairs = [(query, chunks[id]) for id in candidates]
    scores = reranker.predict(pairs)
    scored = [
        (score, id) for score, id in zip(scores, candidates)
    ]

    scored.sort(key=lambda x: x[0], reverse=True)
    return [id for _, id in scored[:top_k]]
